# Course 02: Prepare Data for ML APIs on Google Cloud

**Companion notebook for**: `02-prepare-data-ml-apis.html`

## Learning Objectives
- Call the Cloud Natural Language API for sentiment analysis
- Call the Cloud Vision API for label detection on images
- Use the Cloud Speech-to-Text API to transcribe audio
- Build an Apache Beam pipeline (runs locally or on Dataflow)
- Explore BigQuery data using the `google-cloud-bigquery` SDK

## Prerequisites
- A Google Cloud project with billing enabled (for API calls)
- `GCP_PROJECT_ID` environment variable set, OR replace `your-project-id` below
- Python 3.10+

## Labs Covered
- Vertex AI Qwik Start
- Dataprep, Dataflow (Templates + Python), Dataproc (Console + CLI)
- Cloud NLP API, Speech-to-Text, Video Intelligence

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-aiplatform google-cloud-bigquery google-cloud-language google-cloud-vision google-cloud-speech google-cloud-videointelligence google-cloud-translate apache-beam

In [ ]:
import os
import json

# --- Project Configuration ---
PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
LOCATION = "us-central1"

print(f"Project ID: {PROJECT_ID}")
print(f"Location:   {LOCATION}")

if PROJECT_ID == "your-project-id":
    print("\n** WARNING: Set GCP_PROJECT_ID or replace 'your-project-id' above.")
    print("   Some cells require a real GCP project with billing enabled.")
    print("   Cells that require billing are marked with: REQUIRES: GCP project with billing enabled")

In [ ]:
# Authenticate in Colab (skip if running locally with gcloud auth)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
except ImportError:
    print("Not running in Colab — assuming local gcloud auth.")
    print("Run: gcloud auth application-default login")

---
## Section 1: Cloud Natural Language API — Sentiment Analysis

The **Cloud NLP API** analyzes text for sentiment, entities, syntax, and content classification.
This section demonstrates sentiment analysis on sample text.

**Sentiment scores** range from -1.0 (very negative) to +1.0 (very positive).  
**Magnitude** indicates how much emotional content is present (0 = neutral, higher = more emotional).

In [ ]:
# REQUIRES: GCP project with billing enabled + NL API enabled
# Enable: gcloud services enable language.googleapis.com

from google.cloud import language_v1


def analyze_text_sentiment(text: str) -> dict:
    """
    Analyze sentiment of text using Cloud Natural Language API.
    
    Returns dict with:
        - score: float (-1.0 to 1.0) — overall sentiment
        - magnitude: float (0 to inf) — strength of emotion
        - sentences: list of per-sentence sentiment
    """
    client = language_v1.LanguageServiceClient()
    
    document = language_v1.Document(
        content=text,
        type_=language_v1.Document.Type.PLAIN_TEXT,
    )
    
    response = client.analyze_sentiment(document=document)
    doc_sentiment = response.document_sentiment
    
    return {
        "score": round(doc_sentiment.score, 3),
        "magnitude": round(doc_sentiment.magnitude, 3),
        "sentences": [
            {
                "text": s.text.content[:80],
                "score": round(s.sentiment.score, 3),
                "magnitude": round(s.sentiment.magnitude, 3),
            }
            for s in response.sentences
        ],
    }


def analyze_entities(text: str) -> list:
    """Extract named entities from text using Cloud NLP API."""
    client = language_v1.LanguageServiceClient()
    document = language_v1.Document(
        content=text,
        type_=language_v1.Document.Type.PLAIN_TEXT,
    )
    response = client.analyze_entities(document=document)
    return [
        {
            "name": entity.name,
            "type": entity.type_.name,
            "salience": round(entity.salience, 3),
        }
        for entity in response.entities
    ]


print("NLP helper functions defined: analyze_text_sentiment(), analyze_entities()")

In [ ]:
# --- Sentiment Analysis on Sample Texts ---
# REQUIRES: GCP project with billing enabled

sample_texts = [
    "Google Cloud's Vertex AI platform makes it incredibly easy to deploy machine learning models. The documentation is thorough and the APIs are well-designed.",
    "The service was terrible. Waited 45 minutes, food was cold, and the staff was rude. Never coming back.",
    "The quarterly earnings report shows mixed results. Revenue increased by 12% but operating costs grew faster than expected.",
    "Just finished the GCP MLE certification exam. Some questions were tricky but overall fair. Glad I studied the data pipeline section thoroughly!",
]

try:
    print("Cloud NLP API — Sentiment Analysis Results")
    print("=" * 80)
    
    for i, text in enumerate(sample_texts, 1):
        result = analyze_text_sentiment(text)
        
        # Classify sentiment
        if result["score"] > 0.25:
            label = "POSITIVE"
        elif result["score"] < -0.25:
            label = "NEGATIVE"
        else:
            label = "NEUTRAL/MIXED"
        
        print(f"\nText {i}: \"{text[:70]}...\"")
        print(f"  Overall: {label} (score={result['score']:.3f}, magnitude={result['magnitude']:.3f})")
        print(f"  Sentences:")
        for sent in result["sentences"]:
            print(f"    [{sent['score']:+.2f}] {sent['text']}")
    
    # Also demonstrate entity extraction
    print("\n" + "=" * 80)
    print("Entity Extraction (Text 1):")
    entities = analyze_entities(sample_texts[0])
    for ent in entities:
        print(f"  {ent['name']:<30} {ent['type']:<15} salience={ent['salience']:.3f}")

except Exception as e:
    print(f"NLP API call failed: {e}")
    print("\n--- Mock Output (what you would see with a real GCP project) ---")
    mock_results = [
        ("Google Cloud's Vertex AI platform...", "POSITIVE",  0.850, 1.700),
        ("The service was terrible...",          "NEGATIVE", -0.800, 1.600),
        ("The quarterly earnings report...",     "NEUTRAL",   0.100, 0.800),
        ("Just finished the GCP MLE cert...",    "POSITIVE",  0.500, 1.200),
    ]
    print(f"\n{'Text (truncated)':<45} {'Label':<15} {'Score':>7} {'Magnitude':>10}")
    print("-" * 82)
    for text, label, score, mag in mock_results:
        print(f"{text:<45} {label:<15} {score:>+7.3f} {mag:>10.3f}")
    
    print("\nMock Entity Extraction:")
    print(f"  {'Google Cloud':<30} {'ORGANIZATION':<15} salience=0.450")
    print(f"  {'Vertex AI':<30} {'OTHER':<15} salience=0.320")
    print(f"  {'machine learning':<30} {'OTHER':<15} salience=0.150")
    print(f"  {'models':<30} {'OTHER':<15} salience=0.080")

### Interpreting Sentiment Results

| Score | Magnitude | Interpretation |
|---|---|---|
| +0.8, magnitude 1.5 | Clearly positive with strong emotion |
| -0.7, magnitude 1.2 | Clearly negative with strong emotion |
| +0.1, magnitude 0.3 | Neutral — low emotional content |
| +0.0, magnitude 2.0 | **Mixed** — both positive and negative (high magnitude cancels out score) |

**Key exam insight**: A score near 0 with HIGH magnitude means mixed sentiment, not neutral.

---
## Section 2: Cloud Vision API — Label Detection

The **Cloud Vision API** analyzes images and returns structured labels, text, faces, landmarks, and more.
We will use label detection to identify objects in a publicly hosted sample image.

In [ ]:
# REQUIRES: GCP project with billing enabled + Vision API enabled
# Enable: gcloud services enable vision.googleapis.com

from google.cloud import vision


def detect_labels(image_uri: str, max_results: int = 10) -> list:
    """
    Detect labels in an image using Cloud Vision API.
    
    Args:
        image_uri: Public URL or GCS URI (gs://bucket/path)
        max_results: Max labels to return
    
    Returns:
        List of dicts with 'description' and 'score'
    """
    client = vision.ImageAnnotatorClient()
    image = vision.Image(source=vision.ImageSource(image_uri=image_uri))
    
    response = client.label_detection(image=image, max_results=max_results)
    
    if response.error.message:
        raise Exception(f"Vision API error: {response.error.message}")
    
    return [
        {"description": label.description, "score": round(label.score, 4)}
        for label in response.label_annotations
    ]


def detect_text(image_uri: str) -> str:
    """Detect and extract text (OCR) from an image."""
    client = vision.ImageAnnotatorClient()
    image = vision.Image(source=vision.ImageSource(image_uri=image_uri))
    response = client.text_detection(image=image)
    if response.text_annotations:
        return response.text_annotations[0].description
    return ""


def detect_safe_search(image_uri: str) -> dict:
    """Detect explicit content in an image."""
    client = vision.ImageAnnotatorClient()
    image = vision.Image(source=vision.ImageSource(image_uri=image_uri))
    response = client.safe_search_detection(image=image)
    safe = response.safe_search_annotation
    likelihood_name = (
        "UNKNOWN", "VERY_UNLIKELY", "UNLIKELY", "POSSIBLE", "LIKELY", "VERY_LIKELY"
    )
    return {
        "adult": likelihood_name[safe.adult],
        "violence": likelihood_name[safe.violence],
        "racy": likelihood_name[safe.racy],
        "spoof": likelihood_name[safe.spoof],
        "medical": likelihood_name[safe.medical],
    }


print("Vision helper functions defined: detect_labels(), detect_text(), detect_safe_search()")

In [ ]:
# --- Label Detection on Sample Images ---
# REQUIRES: GCP project with billing enabled

sample_images = [
    {
        "name": "Cat Image",
        "uri": "https://storage.googleapis.com/cloud-samples-data/vision/label/wakeupcat.jpg",
    },
    {
        "name": "City Street",
        "uri": "https://storage.googleapis.com/cloud-samples-data/vision/label/setagaya.jpeg",
    },
]

try:
    for img in sample_images:
        print(f"\nImage: {img['name']}")
        print(f"URI:   {img['uri']}")
        print("-" * 50)
        
        labels = detect_labels(img["uri"])
        print(f"{'Label':<30} {'Confidence':>10}")
        for label in labels:
            bar = '|' * int(label['score'] * 20)
            print(f"  {label['description']:<28} {label['score']:>8.1%}  {bar}")

except Exception as e:
    print(f"Vision API call failed: {e}")
    print("\n--- Mock Output ---")
    print("\nImage: Cat Image")
    mock_cat = [
        ("Cat", 0.983), ("Felidae", 0.961), ("Small to medium-sized cats", 0.945),
        ("Whiskers", 0.932), ("Carnivore", 0.891), ("Tabby cat", 0.867),
    ]
    print(f"{'Label':<30} {'Confidence':>10}")
    for desc, score in mock_cat:
        bar = '|' * int(score * 20)
        print(f"  {desc:<28} {score:>8.1%}  {bar}")
    
    print("\nImage: City Street")
    mock_city = [
        ("Building", 0.971), ("Road", 0.955), ("Urban area", 0.934),
        ("City", 0.912), ("Neighbourhood", 0.889), ("Street", 0.867),
    ]
    print(f"{'Label':<30} {'Confidence':>10}")
    for desc, score in mock_city:
        bar = '|' * int(score * 20)
        print(f"  {desc:<28} {score:>8.1%}  {bar}")

---
## Section 3: Cloud Speech-to-Text — Transcribe Audio

The **Cloud Speech-to-Text API** converts audio to text. It supports:
- **Synchronous**: Audio up to 1 minute, immediate response
- **Asynchronous**: Audio up to 480 minutes, poll for results
- **Streaming**: Real-time transcription

We demonstrate the synchronous API on a short sample.

In [ ]:
# REQUIRES: GCP project with billing enabled + Speech API enabled
# Enable: gcloud services enable speech.googleapis.com

from google.cloud import speech


def transcribe_audio_gcs(gcs_uri: str, language_code: str = "en-US") -> list:
    """
    Transcribe audio from a GCS URI using Cloud Speech-to-Text.
    
    For audio > 1 minute, use long_running_recognize instead.
    
    Args:
        gcs_uri: GCS path to audio file (gs://bucket/file.wav)
        language_code: BCP-47 language code (default: en-US)
    
    Returns:
        List of transcription results with confidence
    """
    client = speech.SpeechClient()
    
    audio = speech.RecognitionAudio(uri=gcs_uri)
    config = speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.FLAC,
        sample_rate_hertz=16000,
        language_code=language_code,
        enable_automatic_punctuation=True,
        enable_word_time_offsets=True,
    )
    
    response = client.recognize(config=config, audio=audio)
    
    results = []
    for result in response.results:
        best = result.alternatives[0]
        results.append({
            "transcript": best.transcript,
            "confidence": round(best.confidence, 4),
            "words": [
                {
                    "word": w.word,
                    "start": w.start_time.total_seconds(),
                    "end": w.end_time.total_seconds(),
                }
                for w in best.words
            ],
        })
    return results


print("Speech-to-Text helper defined: transcribe_audio_gcs()")

In [ ]:
# --- Transcribe a sample audio file from GCS ---
# REQUIRES: GCP project with billing enabled

# Google provides public sample audio files
sample_audio_uri = "gs://cloud-samples-data/speech/brooklyn_bridge.flac"

try:
    results = transcribe_audio_gcs(sample_audio_uri)
    
    print("Speech-to-Text Transcription Results")
    print("=" * 60)
    print(f"Audio: {sample_audio_uri}\n")
    
    for i, result in enumerate(results):
        print(f"Result {i+1}:")
        print(f"  Transcript:  {result['transcript']}")
        print(f"  Confidence:  {result['confidence']:.2%}")
        
        if result["words"]:
            print(f"  Word timestamps:")
            for w in result["words"][:5]:  # Show first 5 words
                print(f"    '{w['word']}' -> {w['start']:.1f}s - {w['end']:.1f}s")
            if len(result["words"]) > 5:
                print(f"    ... and {len(result['words'])-5} more words")

except Exception as e:
    print(f"Speech API call failed: {e}")
    print("\n--- Mock Output ---")
    print("Audio: gs://cloud-samples-data/speech/brooklyn_bridge.flac")
    print("")
    print("Result 1:")
    print("  Transcript:  How old is the Brooklyn Bridge?")
    print("  Confidence:  98.36%")
    print("  Word timestamps:")
    print("    'How' -> 0.0s - 0.3s")
    print("    'old' -> 0.3s - 0.5s")
    print("    'is' -> 0.5s - 0.6s")
    print("    'the' -> 0.6s - 0.7s")
    print("    'Brooklyn' -> 0.7s - 1.1s")

In [ ]:
# --- Long-running transcription pattern (for audio > 1 minute) ---
# REQUIRES: GCP project with billing enabled

"""
# For longer audio files, use long_running_recognize
client = speech.SpeechClient()

audio = speech.RecognitionAudio(uri="gs://my-bucket/long-recording.flac")
config = speech.RecognitionConfig(
    encoding=speech.RecognitionConfig.AudioEncoding.FLAC,
    sample_rate_hertz=44100,
    language_code="en-US",
    enable_automatic_punctuation=True,
    enable_word_time_offsets=True,
    diarization_config=speech.SpeakerDiarizationConfig(
        enable_speaker_diarization=True,
        min_speaker_count=2,
        max_speaker_count=6,
    ),
)

# This returns a long-running operation
operation = client.long_running_recognize(config=config, audio=audio)

# Poll for results (blocks until done)
print("Transcribing... this may take several minutes.")
response = operation.result(timeout=600)  # 10 minute timeout

for result in response.results:
    print(f"Transcript: {result.alternatives[0].transcript}")
    print(f"Confidence: {result.alternatives[0].confidence:.2%}")
"""

print("Long-running transcription pattern (commented out to avoid costs):")
print("  - client.long_running_recognize() for audio > 1 minute")
print("  - operation.result(timeout=600) to poll for results")
print("  - Supports speaker diarization for multi-speaker audio")
print("  - Maximum audio length: 480 minutes")

---
## Section 4: Apache Beam Pipeline (Runs Locally)

**Apache Beam** is the programming model behind Cloud Dataflow. Pipelines written with Beam
can run locally (DirectRunner) or at scale on Dataflow (DataflowRunner).

This section demonstrates a Beam pipeline running locally — no GCP project needed.

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

print(f"Apache Beam version: {beam.__version__}")

In [ ]:
# --- Pipeline 1: Word Count (the "Hello World" of Beam) ---
# This runs LOCALLY with DirectRunner — no GCP project needed

sample_text = [
    "Vertex AI is the unified ML platform on Google Cloud",
    "Dataflow runs Apache Beam pipelines for batch and streaming",
    "BigQuery is a serverless data warehouse on Google Cloud",
    "AutoML trains custom models on Vertex AI without ML expertise",
    "Cloud Vision API detects labels in images on Google Cloud",
]

# Build and run the pipeline
with beam.Pipeline(options=PipelineOptions()) as pipeline:
    word_counts = (
        pipeline
        | "CreateInput" >> beam.Create(sample_text)
        | "SplitWords" >> beam.FlatMap(lambda line: line.lower().split())
        | "FilterStopWords" >> beam.Filter(
            lambda word: word not in {"is", "the", "a", "an", "on", "for", "and", "in", "without"}
        )
        | "PairWithOne" >> beam.Map(lambda word: (word, 1))
        | "GroupAndSum" >> beam.CombinePerKey(sum)
        | "SortByCount" >> beam.Map(lambda kv: (kv[1], kv[0]))  # Swap for sorting
    )
    
    # Print results
    word_counts | "Print" >> beam.Map(lambda kv: print(f"  {kv[1]:<25} count={kv[0]}"))

print("\nPipeline complete! (Ran locally with DirectRunner)")

In [ ]:
# --- Pipeline 2: Data Transformation (ETL pattern) ---
# Simulates processing raw records through a cleaning pipeline

raw_records = [
    {"name": "Alice", "age": 30, "score": 85.5, "department": "Engineering"},
    {"name": "Bob", "age": -1, "score": 92.0, "department": "Sales"},  # Invalid age
    {"name": "Charlie", "age": 25, "score": None, "department": "Engineering"},  # Missing score
    {"name": "Diana", "age": 35, "score": 78.3, "department": "Marketing"},
    {"name": "Eve", "age": 28, "score": 95.1, "department": "Engineering"},
    {"name": "", "age": 40, "score": 88.0, "department": "Sales"},  # Missing name
    {"name": "Frank", "age": 32, "score": 71.2, "department": "Marketing"},
]


def validate_record(record):
    """Filter out invalid records."""
    if not record.get("name"):
        return False
    if record.get("age", -1) < 0:
        return False
    if record.get("score") is None:
        return False
    return True


def enrich_record(record):
    """Add computed fields."""
    record["score_category"] = (
        "Excellent" if record["score"] >= 90
        else "Good" if record["score"] >= 80
        else "Average" if record["score"] >= 70
        else "Below Average"
    )
    record["name_upper"] = record["name"].upper()
    return record


print("ETL Pipeline: Raw Records -> Validate -> Enrich -> Output")
print(f"Input: {len(raw_records)} records")
print()

with beam.Pipeline(options=PipelineOptions()) as pipeline:
    cleaned = (
        pipeline
        | "CreateRecords" >> beam.Create(raw_records)
        | "ValidateRecords" >> beam.Filter(validate_record)
        | "EnrichRecords" >> beam.Map(enrich_record)
        | "FormatOutput" >> beam.Map(
            lambda r: f"  {r['name_upper']:<12} age={r['age']:<4} "
                      f"score={r['score']:<6.1f} -> {r['score_category']:<15} "
                      f"dept={r['department']}"
        )
        | "Print" >> beam.Map(print)
    )

print(f"\nOutput: Records that passed validation and enrichment")
print("(Records with missing name, negative age, or null score were filtered out)")

In [ ]:
# --- Pipeline 3: GroupByKey pattern (aggregation) ---
# This pattern is the basis for many Dataflow use cases

sales_data = [
    ("Electronics", 1200), ("Clothing", 350), ("Electronics", 800),
    ("Food", 150), ("Clothing", 275), ("Electronics", 950),
    ("Food", 200), ("Clothing", 420), ("Food", 180),
    ("Electronics", 1100), ("Clothing", 310),
]

print("Aggregation Pipeline: Sales by Category")
print("=" * 50)

with beam.Pipeline(options=PipelineOptions()) as pipeline:
    (
        pipeline
        | "CreateSales" >> beam.Create(sales_data)
        | "SumByCategory" >> beam.CombinePerKey(sum)
        | "FormatResults" >> beam.Map(
            lambda kv: print(f"  {kv[0]:<15} Total: ${kv[1]:>8,}")
        )
    )

print("\nThis is the same pattern used in Dataflow for large-scale aggregations.")
print("Replace DirectRunner with DataflowRunner to run on GCP.")

In [ ]:
# --- How to run on Dataflow (reference code, not executable here) ---

dataflow_code = """
# To run this pipeline on Cloud Dataflow instead of locally:

from apache_beam.options.pipeline_options import PipelineOptions, GoogleCloudOptions

options = PipelineOptions([
    '--runner=DataflowRunner',
    '--project=my-gcp-project',
    '--region=us-central1',
    '--temp_location=gs://my-bucket/temp',
    '--staging_location=gs://my-bucket/staging',
    '--job_name=my-etl-pipeline',
])

with beam.Pipeline(options=options) as p:
    (
        p
        | beam.io.ReadFromBigQuery(
            query='SELECT * FROM `project.dataset.table`',
            use_standard_sql=True,
        )
        | beam.Filter(validate_record)
        | beam.Map(enrich_record)
        | beam.io.WriteToBigQuery(
            'project:dataset.output_table',
            schema='name:STRING,age:INTEGER,score:FLOAT,category:STRING',
            write_disposition=beam.io.BigQueryDisposition.WRITE_TRUNCATE,
        )
    )
"""

print("=" * 60)
print("Running on Cloud Dataflow (reference code)")
print("=" * 60)
print(dataflow_code)
print("Key changes from local to Dataflow:")
print("  1. --runner=DataflowRunner (instead of DirectRunner)")
print("  2. --project, --region, --temp_location are required")
print("  3. I/O uses BigQuery, GCS, or Pub/Sub (instead of in-memory)")
print("  4. Workers scale automatically based on data volume")

---
## Section 5: BigQuery Data Exploration

**BigQuery** is Google Cloud's serverless data warehouse. It is the most common data source
for ML on GCP. The `google-cloud-bigquery` SDK lets you query BigQuery from Python.

We will query a public dataset to demonstrate the SDK pattern.

In [ ]:
# REQUIRES: GCP project with billing enabled (for BigQuery queries)
# Public datasets are free to query (up to 1TB/month)

from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)
print(f"BigQuery client initialized for project: {PROJECT_ID}")

In [ ]:
# --- Query a public dataset: Stack Overflow posts ---
# REQUIRES: GCP project with billing enabled
# This query processes ~100MB (well within free tier)

query = """
SELECT
    EXTRACT(YEAR FROM creation_date) AS year,
    COUNT(*) AS num_questions,
    ROUND(AVG(answer_count), 2) AS avg_answers,
    ROUND(AVG(score), 2) AS avg_score
FROM `bigquery-public-data.stackoverflow.posts_questions`
WHERE
    tags LIKE '%machine-learning%'
    AND creation_date >= '2015-01-01'
GROUP BY year
ORDER BY year
"""

try:
    print("Querying BigQuery: Stack Overflow ML questions by year...")
    print(f"SQL:\n{query}")
    
    df = client.query(query).to_dataframe()
    
    print("\nResults:")
    print(df.to_string(index=False))
    
    print(f"\nTotal ML questions since 2015: {df['num_questions'].sum():,}")
    print(f"Bytes processed: shown in BigQuery console")

except Exception as e:
    print(f"BigQuery query failed: {e}")
    print("\n--- Mock Output ---")
    print(f"{'year':<6} {'num_questions':>14} {'avg_answers':>12} {'avg_score':>10}")
    print("-" * 46)
    mock_data = [
        (2015, 8234, 1.45, 1.23), (2016, 12456, 1.38, 1.15),
        (2017, 18923, 1.32, 1.08), (2018, 24567, 1.28, 0.95),
        (2019, 28901, 1.22, 0.88), (2020, 31245, 1.18, 0.82),
        (2021, 29876, 1.15, 0.78), (2022, 27654, 1.12, 0.75),
        (2023, 25432, 1.08, 0.71), (2024, 22100, 1.05, 0.68),
    ]
    for year, nq, aa, asc in mock_data:
        print(f"{year:<6} {nq:>14,} {aa:>12.2f} {asc:>10.2f}")

In [ ]:
# --- BigQuery ML pattern (reference — train a model with SQL) ---

bqml_example = """
-- BigQuery ML: Train a logistic regression model with SQL
-- No Python, no infrastructure, no ML expertise needed

CREATE OR REPLACE MODEL `my_project.my_dataset.churn_model`
OPTIONS(
    model_type = 'LOGISTIC_REG',
    input_label_cols = ['churned'],
    auto_class_weights = TRUE,
    max_iterations = 20
) AS
SELECT
    tenure_months,
    monthly_charges,
    total_charges,
    contract_type,
    internet_service,
    payment_method,
    churned  -- target label
FROM `my_project.my_dataset.customer_data`
WHERE split = 'train';

-- Evaluate the model
SELECT * FROM ML.EVALUATE(MODEL `my_project.my_dataset.churn_model`);

-- Make predictions
SELECT *
FROM ML.PREDICT(
    MODEL `my_project.my_dataset.churn_model`,
    (SELECT * FROM `my_project.my_dataset.customer_data` WHERE split = 'test')
);
"""

print("BigQuery ML — Train ML Models with SQL")
print("=" * 60)
print(bqml_example)
print("Supported model types:")
print("  - LOGISTIC_REG (classification)")
print("  - LINEAR_REG (regression)")
print("  - KMEANS (clustering)")
print("  - MATRIX_FACTORIZATION (recommendations)")
print("  - BOOSTED_TREE_CLASSIFIER / REGRESSOR (XGBoost)")
print("  - DNN_CLASSIFIER / REGRESSOR (Deep Neural Network)")
print("  - TENSORFLOW (import SavedModel)")
print("  - AUTOML_CLASSIFIER / REGRESSOR (AutoML Tables)")

---
## Section 6: Service Decision Quick Reference

Use this as a study aid for the exam — match the scenario to the right service.

In [ ]:
# Quick reference decision matrix for data tools and APIs

data_tool_scenarios = [
    ("Build a new serverless ETL pipeline", "Dataflow", "Serverless, unified batch/stream"),
    ("Migrate existing Spark jobs to GCP", "Dataproc", "Managed Spark, lift-and-shift"),
    ("Business analyst needs to clean a CSV", "Dataprep", "Visual UI, no code"),
    ("Real-time event processing from Pub/Sub", "Dataflow", "Streaming with Apache Beam"),
    ("Run Spark MLlib on GCP", "Dataproc", "Spark ecosystem"),
    ("Transform data already in BigQuery", "BigQuery SQL", "In-warehouse transforms"),
    ("Orchestrate an ML training pipeline", "Vertex AI Pipelines", "ML workflow orchestration"),
]

api_scenarios = [
    ("Detect objects in images", "Cloud Vision API", "label_detection"),
    ("Extract text from scanned documents", "Cloud Vision API", "DOCUMENT_TEXT_DETECTION"),
    ("Analyze customer review sentiment", "Cloud NLP API", "analyze_sentiment"),
    ("Extract named entities from text", "Cloud NLP API", "analyze_entities"),
    ("Transcribe meeting recordings", "Speech-to-Text", "long_running_recognize"),
    ("Real-time phone call transcription", "Speech-to-Text", "streaming_recognize"),
    ("Translate product descriptions", "Cloud Translation", "translate_text"),
    ("Detect explicit content in videos", "Video Intelligence", "EXPLICIT_CONTENT_DETECTION"),
    ("Track objects across video frames", "Video Intelligence", "OBJECT_TRACKING"),
    ("Detect scene changes in video", "Video Intelligence", "SHOT_CHANGE_DETECTION"),
]

print("DATA TOOL DECISION MATRIX")
print("=" * 75)
print(f"{'Scenario':<45} {'Service':<18} {'Why'}")
print("-" * 75)
for scenario, service, reason in data_tool_scenarios:
    print(f"{scenario:<45} {service:<18} {reason}")

print(f"\n\nAPI SELECTION MATRIX")
print("=" * 75)
print(f"{'Scenario':<45} {'API':<18} {'Method/Feature'}")
print("-" * 75)
for scenario, api, method in api_scenarios:
    print(f"{scenario:<45} {api:<18} {method}")

---
## Summary

In this notebook we covered the core concepts and SDK patterns for Course 02:

1. **Cloud NLP API** — Sentiment analysis and entity extraction on text. Score + magnitude interpretation.

2. **Cloud Vision API** — Label detection, OCR, and safe search on images. Send a URI, get annotations.

3. **Cloud Speech-to-Text** — Transcribe audio (sync for < 1 min, async for longer). Speaker diarization.

4. **Apache Beam Pipelines** — Word count, ETL, and aggregation patterns running locally with DirectRunner. Same code runs on Dataflow with `--runner=DataflowRunner`.

5. **BigQuery Exploration** — Query public datasets, understand BigQuery ML pattern for training models with SQL.

**Key exam takeaways**:
- **Dataflow** = default for serverless ETL (batch + streaming)
- **Dataproc** = existing Spark/Hadoop code
- **Dataprep** = non-technical, visual, ad-hoc
- Match data type (image/text/audio/video) to the corresponding pre-trained API

**Next notebook**: Course 03 covers Big Data and ML Fundamentals on Google Cloud.